In [ ]:
from pathlib import Path
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from dotenv import load_dotenv
from pyspark.ml.linalg import SparseVector, VectorUDT
from pyspark.ml.regression import LinearRegression
from pyspark import StorageLevel
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

load_dotenv()

In [ ]:
spark = (
    SparkSession.builder
    .appName("uk-property-flip-detector")
    .config("spark.jars", "jars/postgresql-jdbc.jar")
    .config("spark.driver.memory", "6g")
    .getOrCreate()
)

In [ ]:
BASE_YEAR = 1995

df = (
    spark.read
    .format("jdbc")
    .option("url", "jdbc:postgresql://192.168.0.204:5432/land_registry")
    .option("dbtable", "repeat_sale_pairs_regression")
    .option("user", os.environ["PGUSER"])
    .option("password", os.environ["PGPASSWORD"])
    .option("driver", "org.postgresql.Driver")
    .option("partitionColumn", "property_id")
    .option("lowerBound", "1")
    .option("upperBound", "15184630")
    .option("numPartitions", "8")
    .load()
)

pairs = (
    df
    .withColumn(
        # month index - diff in months from base + months elapsed from last
        "prev_period",
        (F.year("prev_date_of_transfer") - BASE_YEAR) * 12
        + (F.month("prev_date_of_transfer") - 1),
    )
    .withColumn(
        "curr_period",
        (F.year("curr_date_of_transfer") - BASE_YEAR) * 12
        + (F.month("curr_date_of_transfer") - 1),
    )
    # cast decimal as double
    .withColumn("log_price_ratio", F.col("log_price_ratio").cast("double"))
)


pairs.select(
    "prev_date_of_transfer", "prev_period",
    "curr_date_of_transfer", "curr_period",
).show(10)

### Parallel JDBC read

Parallelism isn't strictly needed at this scale (12.4M rows fits comfortably on one machine), but the read is partitioned to demonstrate how it would scale.

Spark splis range into 8 slices and sends 8 queries to Postgres, one per partition, each of the form:

    SELECT <columns> FROM repeat_sale_pairs_regression
    WHERE property_id >= X AND property_id < Y

Each executor thread reads its slice concurrently. The bounds only control how the range is divided; they don't filter rows.

`.explain(True)` demonstrates the optimisations done by the Catalyst Optimiser when calling select on the two cols.

In [ ]:
pairs.select("prev_period", "curr_period").explain(True)

In [ ]:
# period index validation
pairs.select(
    F.min("prev_period"), F.max("prev_period"),
    F.min("curr_period"), F.max("curr_period"),
).show()

print("backwards:", pairs.filter(F.col("curr_period") < F.col("prev_period")).count())
print("same month:", pairs.filter(F.col("curr_period") == F.col("prev_period")).count())

Trimming and filtering: drops July 2026 (incomplete registrations) and same-month pairs (zero rows in the design matrix); keeps only regression columns before caching.

In [ ]:
MAX_PERIOD = 377  # July 2026 (378) trimmed: incomplete due to registration lag

pairs_clean = (
    pairs
    .filter(F.col("curr_period") <= MAX_PERIOD)
    .filter(F.col("curr_period") != F.col("prev_period"))
    .select("property_id", "prev_period", "curr_period", "log_price_ratio")
    .cache()
)

print("pairs after filtering:", pairs_clean.count())

print("overlap:", pairs.filter(
    (F.col("curr_period") == 378) & (F.col("curr_period") == F.col("prev_period"))
).count())

In [ ]:
pairs_clean.count()

In [ ]:
N_PARAMS = MAX_PERIOD

# user defined function declaring return type to be VectorUDT from ml.linalg
# Apache Arrow turned off due to incompatibility with user defined types
@F.udf(returnType=VectorUDT(), useArrow=False)
def bmn_vector(prev_p, curr_p):
    entries = {}
    if prev_p > 0:
        entries[prev_p - 1] = -1.0
    if curr_p > 0:
        entries[curr_p - 1] = 1.0
    return SparseVector(N_PARAMS, entries)

reg_input = (
    pairs_clean
    .withColumn("features", bmn_vector("prev_period", "curr_period"))
    .withColumn("interval", (F.col("curr_period") - F.col("prev_period")).cast("double"))
    .select("features", F.col("log_price_ratio").alias("y"), "interval")
    .persist(StorageLevel.MEMORY_AND_DISK)
)
reg_input.count()  # materialises the cache; expect 12,407,196

lr = LinearRegression(
    featuresCol="features",
    labelCol="y",
    fitIntercept=False,
    solver="normal",
    regParam=0.0,
    standardization=False,
)
model = lr.fit(reg_input)

In [ ]:
beta = np.concatenate([[0.0], model.coefficients.toArray()])
index = 100 * np.exp(beta)
dates = pd.date_range("1995-01-01", periods=len(index), freq="MS")
bmn = pd.Series(index, index=dates, name="BMN index")

print(len(index))     # 378: periods 0..377
print(index[:13])     # 1995 through Jan 1996
print(index[-1])      # June 2026 
print(model.summary.r2, model.summary.rootMeanSquaredError)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
bmn.plot(ax=ax)
ax.set_title("Repeat-sales house price index (plain BMN), Jan 1995 = 100")
ax.set_ylabel("Index")
ax.grid(alpha=0.3)
plt.show()

In [ ]:
resid = (
    model.transform(reg_input)
    .withColumn("e2", (F.col("y") - F.col("prediction")) ** 2)
)

s = resid.agg(
    F.covar_samp("interval", "e2").alias("cov"),
    F.var_samp("interval").alias("var"),
    F.avg("interval").alias("mean_x"),
    F.avg("e2").alias("mean_e2"),
).first()

b = s["cov"] / s["var"]
a = s["mean_e2"] - b * s["mean_x"]
print(f"a = {a:.5f}, b = {b:.6f}")

In [ ]:
weighted = reg_input.withColumn(
    "w", 1.0 / (F.lit(a) + F.lit(b) * F.col("interval"))
)

lr_cs = LinearRegression(
    featuresCol="features",
    labelCol="y",
    weightCol="w",
    fitIntercept=False,
    solver="normal",
    regParam=0.0,
    standardization=False,
)
model_cs = lr_cs.fit(weighted)

beta_cs = np.concatenate([[0.0], model_cs.coefficients.toArray()])
index_cs = 100 * np.exp(beta_cs)
bmn_cs = pd.Series(index_cs, index=dates, name="Case-Shiller index")
print(index_cs[-1])

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
bmn.plot(ax=ax)
bmn_cs.plot(ax=ax)
ax.set_title("Repeat-sales index: plain BMN vs Case-Shiller weighted, Jan 1995 = 100")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

In [ ]:
(
    resid
    .withColumn("years", F.floor(F.col("interval") / 12))
    .groupBy("years")
    .agg(F.avg("e2").alias("mean_e2"), F.count("*").alias("n"))
    .orderBy("years")
    .show(35)
)

In [ ]:
def fit_bmn(df, weight_col=None):
    params = dict(featuresCol="features", labelCol="y", fitIntercept=False,
                  solver="normal", regParam=0.0, standardization=False)
    if weight_col:
        params["weightCol"] = weight_col
    return LinearRegression(**params).fit(df)

def to_index(model, name):
    beta = np.concatenate([[0.0], model.coefficients.toArray()])
    return pd.Series(100 * np.exp(beta), index=dates, name=name)

MIN_INTERVAL = 24  # months
reg_long = reg_input.filter(F.col("interval") >= MIN_INTERVAL)

# stage 1
m1 = fit_bmn(reg_long)

# stage 2
r = m1.transform(reg_long).withColumn("e2", (F.col("y") - F.col("prediction")) ** 2)
s = r.agg(F.covar_samp("interval", "e2").alias("cov"), F.var_samp("interval").alias("var"),
          F.avg("interval").alias("mx"), F.avg("e2").alias("mz")).first()
b2 = s["cov"] / s["var"]
a2 = s["mz"] - b2 * s["mx"]
print(f"a = {a2:.5f}, b = {b2:.6f}")

# stage 3
m3 = fit_bmn(reg_long.withColumn("w", 1.0 / (F.lit(a2) + F.lit(b2) * F.col("interval"))), "w")
bmn_cs_long = to_index(m3, "Case-Shiller, holds ≥ 2y")
print(bmn_cs_long.iloc[-1])

In [ ]:
g = lambda s: 100 * (s / s.shift(12) - 1)

fig, ax = plt.subplots(figsize=(11, 4))
(g(bmn_cs) - g(bmn)).plot(ax=ax, label="Case-Shiller (all holds)")
(g(bmn_cs_long) - g(bmn)).plot(ax=ax, label=f"Case-Shiller (holds ≥ {MIN_INTERVAL // 12}y)")
ax.axhline(0, color="grey", linewidth=0.8)
ax.set_title("Difference in 12-month growth vs plain BMN (percentage points)")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

In [ ]:
def save_index(series, out_dir, filename):
    return series.rename_axis("month").to_frame().to_csv(out_dir / f"{filename}")

out = Path("../data/processed")
out.mkdir(parents=True, exist_ok=True)

save_index(bmn, out, "bmn_index_plain.csv")
save_index(bmn_cs, out, "bmn_index_case_shiller.csv")
save_index(bmn_cs_long, out, f"bmn_index_case_shiller_min{MIN_INTERVAL}m.csv")
reg_input.unpersist()